# Tutorial 03: Context and Memory - Personalized Travel Assistant

## 📚 Learning Objectives
By the end of this notebook, you will be able to:
- Understand the difference between threads (conversation history) and memory (user knowledge)
- Create a custom Context Provider (`BaseContextProvider`) that adds persistent memory
- Automatically extract and store user preferences
- Build a personalized agent that remembers user details across sessions
- Serialize and restore memory state

## 🎯 Key Concepts

### Thread vs Memory: What's the Difference?

**Thread (from Tutorial 03):**
- Stores conversation history ("I said X, you said Y")
- Ephemeral: suited for a single conversation session
- Example: "User asked about Paris, then asked about the weather there"

**Memory (this tutorial):**
- Stores facts about the user ("User prefers beach vacations")
- Persistent: survives across sessions and threads
- Example: "User's name is Sarah, budget-conscious, vegetarian"

### Context Provider (`BaseContextProvider`)

**Context Provider** is a component that:
1. **Listens** to conversations (via `after_run()` method)
2. **Extracts** important information (user preferences, facts)
3. **Provides context** before each AI call (via `before_run()` method)
4. **Persists** across sessions (via `serialize()` method)

```python
class CustomMemory(BaseContextProvider):
    async def after_run(self, *, agent, session, context, state):
        # Called after the agent responds
        # Extract information from context.input_messages
    
    async def before_run(self, *, agent, session, context, state):
        # Called before the agent responds
        # Provide remembered context via context.extend_instructions()
    
    def serialize(self):
        # Save memory state
```

---

## Step 1: Setup and Imports

In [ ]:
import asyncio
import json
from collections.abc import MutableSequence, Sequence
from typing import Annotated, Any
from random import randint, choice

from agent_framework import (
    Agent,
    AgentSession,
    BaseContextProvider,
    ChatOptions,
    Message,
    SessionContext,
    SupportsAgentRun,
    SupportsChatGetResponse,
)
from agent_framework.azure import AzureOpenAIChatClient
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import DefaultAzureCredential
from azure.identity.aio import AzureCliCredential, DefaultAzureCredential

from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()
print("✅ Imports successful!")

In [ ]:
import os

mcp_server_uri = os.getenv("MCP_SERVER_URI")
azure_openai_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# === Select Authentication Method ===
# True: API key authentication, False: DefaultAzureCredential authentication
USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")

if USE_KEY_AUTH:
    print("🔑 Authentication: API Key")
else:
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    print("🔐 Authentication: DefaultAzureCredential")

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

# NOTE: Keep the Foundry conversation client and the structured output client for memory extraction separate.
# The Foundry side may have unstable response_format, so use Azure OpenAI Chat for extraction.
def create_azure_openai_chat_client(credential):
    return AzureAIAgentClient(
        credential=credential,
        project_endpoint=project_endpoint,
        model_deployment_name=model_deployment,
    )

def create_azure_openai_chat_client(credential=None):
    client_kwargs = {
        "endpoint": azure_openai_endpoint,
        "api_version": api_version,
        "deployment_name": azure_deployment,
    }
    if USE_KEY_AUTH:
        client_kwargs["api_key"] = azure_openai_key
    else:
        client_kwargs["credential"] = credential
    return AzureOpenAIChatClient(**client_kwargs)


def create_memory_extraction_client(credential=None):
    return create_azure_openai_chat_client(credential)


print(f"MCP Server URI: {mcp_server_uri}")
print(f"Foundry Project Endpoint: {project_endpoint}")
print(f"Foundry Model Deployment: {model_deployment}")
print(f"Azure OpenAI Endpoint: {azure_openai_endpoint}, Deployment: {azure_deployment}")

## Step 2: The Problem - An Agent Without Memory

First, let's see the limitation: even with threads, the agent doesn't remember user details across sessions.

In [ ]:
async def agent_without_memory():
    """
    Demonstrates that sessions remember conversations but not facts about the user.
    """
    print("=== Agent Without Memory (Session Only) ===")
    print("Sessions remember conversations but not facts about the user.\n")

    if USE_KEY_AUTH:
        foundry_client = create_azure_openai_chat_client()
        async with Agent(
            client=foundry_client,
            name="BasicTravelAgent",
            instructions="You are a travel assistant. Help users plan their trips.",
        ) as agent:
            session1 = agent.create_session()

            response1 = await agent.run("My name is Sarah and I love beach vacations.", session=session1)
            print("User: My name is Sarah and I love beach vacations.")
            print(f"Agent: {response1.text}\n")

            print("--- Session 2 (New Session) ---")
            session2 = agent.create_session()

            response2 = await agent.run("Can you recommend a destination?", session=session2)
            print("User: Can you recommend a destination?")
            print(f"Agent: {response2.text}\n")

            print("❌ Problem: The agent forgot Sarah's name and beach preference!")
            print("   Even if we reuse session1, preferences are not 'remembered' as facts.\n")
            return

    async with DefaultAzureCredential() as credential:
        foundry_client = create_azure_openai_chat_client(credential)
        async with Agent(
            client=foundry_client,
            name="BasicTravelAgent",
            instructions="You are a travel assistant. Help users plan their trips.",
        ) as agent:
            session1 = agent.create_session()

            response1 = await agent.run("My name is Sarah and I love beach vacations.", session=session1)
            print("User: My name is Sarah and I love beach vacations.")
            print(f"Agent: {response1.text}\n")

            print("--- Session 2 (New Session) ---")
            session2 = agent.create_session()

            response2 = await agent.run("Can you recommend a destination?", session=session2)
            print("User: Can you recommend a destination?")
            print(f"Agent: {response2.text}\n")

            print("❌ Problem: The agent forgot Sarah's name and beach preference!")
            print("   Even if we reuse session1, preferences are not 'remembered' as facts.\n")

await agent_without_memory()

## Step 3: Define the User Profile Model

First, let's define what we want to remember about the user.

In [ ]:
class TravelProfile(BaseModel):
    """User travel preferences and profile."""

    name: str | None = Field(None, description="The user's name")
    preferred_destinations: list[str] = Field(
        default_factory=list,
        description="Types of destinations the user prefers (e.g., beach, mountain, city, etc.)",
    )
    budget_level: str | None = Field(
        None,
        description="Budget level (e.g., budget/moderate/luxury)",
    )
    dietary_restrictions: list[str] = Field(
        default_factory=list,
        description="Dietary restrictions (e.g., vegetarian, vegan, halal, kosher, etc.)",
    )
    interests: list[str] = Field(
        default_factory=list,
        description="Travel interests (e.g., history, food, adventure, relaxation, etc.)",
    )


print("✅ TravelProfile model defined")
print("\nExample profile:")
example = TravelProfile(
    name="Sarah",
    preferred_destinations=["beach", "tropical"],
    budget_level="moderate",
    dietary_restrictions=["vegetarian"],
    interests=["relaxation", "food"],
)
print(example.model_dump_json(indent=2))

## Step 4: Create a Context Provider with Memory

Now let's build the memory component!

In [ ]:
import re

class TravelMemory(BaseContextProvider):
    """
    A Context Provider that remembers user travel preferences.

    Implementation following the official sample simple_context_provider.py:
    1. Listen to conversations and extract user information (after_run)
    2. Provide remembered context before each agent response (before_run)
    3. Persist user profile across sessions
    """

    def __init__(
        self,
        chat_client: SupportsChatGetResponse,
        profile: TravelProfile | None = None,
        source_id: str = "travel_memory",
        **kwargs: Any,
    ):
        super().__init__(source_id)
        self._chat_client = chat_client

        if profile:
            self.profile = profile
        elif kwargs:
            self.profile = TravelProfile.model_validate(kwargs)
        else:
            self.profile = TravelProfile()

    @staticmethod
    def _extract_json(text: str) -> str:
        md_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
        if md_match:
            return md_match.group(1)
        brace_match = re.search(r'\{.*\}', text, re.DOTALL)
        if brace_match:
            return brace_match.group(0)
        return ""

    @staticmethod
    def _has_updates(profile: TravelProfile) -> bool:
        return any(
            [
                bool(profile.name),
                bool(profile.preferred_destinations),
                bool(profile.budget_level),
                bool(profile.dietary_restrictions),
                bool(profile.interests),
            ]
        )

    async def after_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        """Extract travel profile from the most recent user input."""
        user_messages = [
            msg for msg in context.input_messages
            if hasattr(msg, "role") and msg.role == "user"
        ]
        if not user_messages:
            return

        try:
            result = await self._chat_client.get_response(
                messages=user_messages,
                options={
                    "instructions": """
                    Extract only travel profile updates from the following user input.
                    - Extract: name, destination preferences, budget level, dietary restrictions, interests
                    - Only extract fields that are explicitly mentioned
                    - Set unmentioned fields to null or empty list
                    - If there is no new profile information at all, return an empty JSON object {}
                    - Return only JSON, no explanations or markdown
                    """,
                    "response_format": TravelProfile,
                },
            )
        except Exception:
            return

        extracted: TravelProfile | None = None
        try:
            extracted = result.value
        except Exception:
            raw_json = self._extract_json(result.text)
            if not raw_json or raw_json == "{}":
                return
            try:
                extracted = TravelProfile.model_validate_json(raw_json)
            except Exception:
                return

        if not extracted or not self._has_updates(extracted):
            return

        if extracted.name:
            self.profile.name = extracted.name

        if extracted.preferred_destinations:
            for dest in extracted.preferred_destinations:
                if dest not in self.profile.preferred_destinations:
                    self.profile.preferred_destinations.append(dest)

        if extracted.budget_level:
            self.profile.budget_level = extracted.budget_level

        if extracted.dietary_restrictions:
            for restriction in extracted.dietary_restrictions:
                if restriction not in self.profile.dietary_restrictions:
                    self.profile.dietary_restrictions.append(restriction)

        if extracted.interests:
            for interest in extracted.interests:
                if interest not in self.profile.interests:
                    self.profile.interests.append(interest)

        print(f"  📝 [Memory] Extracted: {extracted.model_dump()}")

    async def before_run(
        self,
        *,
        agent: Any,
        session: AgentSession | None,
        context: SessionContext,
        state: dict[str, Any],
    ) -> None:
        memory_lines: list[str] = []

        if self.profile.name:
            memory_lines.append(f"User name: {self.profile.name}")
        if self.profile.preferred_destinations:
            memory_lines.append(f"Preferred destinations: {', '.join(self.profile.preferred_destinations)}")
        if self.profile.budget_level:
            memory_lines.append(f"Budget level: {self.profile.budget_level}")
        if self.profile.dietary_restrictions:
            memory_lines.append(f"Dietary restrictions: {', '.join(self.profile.dietary_restrictions)}")
        if self.profile.interests:
            memory_lines.append(f"Interests: {', '.join(self.profile.interests)}")

        if not memory_lines:
            return

        memory_context = (
            "What I remember about this user: "
            + ", ".join(memory_lines)
            + ". Always consider this information, use their name, and make personalized suggestions."
        )

        print(f"🔍 [Context Provider] Injecting memory: {self.profile.name}")
        context.extend_instructions(self.source_id, memory_context)

    def serialize(self) -> str:
        return self.profile.model_dump_json()


print("✅ TravelMemory context provider created!")

## Step 5: Agent with Memory - First Conversation

Let's see memory in action!

In [ ]:
async def agent_with_memory_first_session():
    """
    First session: The agent learns about the user.
    """
    print("=== Agent with Memory - Learning About the User ===")

    if USE_KEY_AUTH:
        foundry_client = create_azure_openai_chat_client()
        memory = TravelMemory(foundry_client)

        async with Agent(
            client=foundry_client,
            name="MemoryEnabledTravelAgent",
            instructions="""
            You are a personalized travel assistant.
            Always address the user by name when you know it.
            Tailor your suggestions based on the user's preferences.
            """,
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()

            query1 = "Hi! My name is Sarah and I love beach vacations."
            print(f"\nUser: {query1}")
            response1 = await agent.run(query1, session=session)
            print(f"Agent: {response1.text}")

            query2 = "My budget is moderate, not too expensive."
            print(f"\nUser: {query2}")
            response2 = await agent.run(query2, session=session)
            print(f"Agent: {response2.text}")

            query3 = "Also, I'm vegetarian so food options are important."
            print(f"\nUser: {query3}")
            response3 = await agent.run(query3, session=session)
            print(f"Agent: {response3.text}")

            print("\n" + "=" * 60)
            print("What Memory Learned:")
            print("=" * 60)
            print(f"Name: {memory.profile.name}")
            print(f"Preferred destinations: {memory.profile.preferred_destinations}")
            print(f"Budget level: {memory.profile.budget_level}")
            print(f"Dietary restrictions: {memory.profile.dietary_restrictions}")
            print(f"Interests: {memory.profile.interests}")

            saved_memory = memory.serialize()
            print("\n💾 Serialized memory for persistence:")
            print(saved_memory)
            return saved_memory

    async with DefaultAzureCredential() as credential:
        foundry_client = create_azure_openai_chat_client(credential)
        memory = TravelMemory(foundry_client)

        async with Agent(
            client=foundry_client,
            name="MemoryEnabledTravelAgent",
            instructions="""
            You are a personalized travel assistant.
            Always address the user by name when you know it.
            Tailor your suggestions based on the user's preferences.
            """,
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()

            query1 = "Hi! My name is Sarah and I love beach vacations."
            print(f"\nUser: {query1}")
            response1 = await agent.run(query1, session=session)
            print(f"Agent: {response1.text}")

            query2 = "My budget is moderate, not too expensive."
            print(f"\nUser: {query2}")
            response2 = await agent.run(query2, session=session)
            print(f"Agent: {response2.text}")

            query3 = "Also, I'm vegetarian so food options are important."
            print(f"\nUser: {query3}")
            response3 = await agent.run(query3, session=session)
            print(f"Agent: {response3.text}")

            print("\n" + "=" * 60)
            print("What Memory Learned:")
            print("=" * 60)
            print(f"Name: {memory.profile.name}")
            print(f"Preferred destinations: {memory.profile.preferred_destinations}")
            print(f"Budget level: {memory.profile.budget_level}")
            print(f"Dietary restrictions: {memory.profile.dietary_restrictions}")
            print(f"Interests: {memory.profile.interests}")

            saved_memory = memory.serialize()
            print("\n💾 Serialized memory for persistence:")
            print(saved_memory)
            return saved_memory


saved_memory = await agent_with_memory_first_session()

In [ ]:
saved_memory

## Step 6: Resume with Saved Memory - Interactive Chat

Now let's create a new agent with saved memory - simulating a new session.

**Note**: We've improved the Context Provider to work correctly. Before running the cell below, **re-run the TravelMemory class definition cell**.


In [ ]:
async def agent_with_restored_memory(saved_memory_json: str):
    """
    New session: The agent restores memory and remembers the user!
    """
    print("\n=== Agent with Memory - New Session (Restored Memory) ===")
    print("Simulating a completely new session with saved memory...\n")

    profile_dict = json.loads(saved_memory_json)
    restored_profile = TravelProfile.model_validate(profile_dict)

    print("✅ Restored profile:")
    print(f"   Name: {restored_profile.name}")
    print(f"   Preferences: {restored_profile.preferred_destinations}")
    print(f"   Budget: {restored_profile.budget_level}\n")

    if USE_KEY_AUTH:
        foundry_client = create_azure_openai_chat_client()
        memory = TravelMemory(foundry_client, profile=restored_profile)

        async with Agent(
            client=foundry_client,
            name="MemoryEnabledTravelAgent_Session2",
            instructions="""
            You are a personalized travel assistant.
            Always address the user by name when you know it.
            Tailor your suggestions based on the user's preferences.
            """,
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()
            query = "Can you recommend my next travel destination?"
            print(f"User: {query}")
            response = await agent.run(query, session=session)
            print(f"Agent: {response.text}\n")

            print("=" * 60)
            print("🎉 Success!")
            print("=" * 60)
            print("Notice that the agent:")
            print("1. Used Sarah's name")
            print("2. Recommended beach destinations")
            print("3. Considered moderate budget")
            print("4. Mentioned vegetarian food options")
            print("\nAll from a different session!")
            return

    async with DefaultAzureCredential() as credential:
        foundry_client = create_azure_openai_chat_client(credential)
        memory = TravelMemory(foundry_client, profile=restored_profile)

        async with Agent(
            client=foundry_client,
            name="MemoryEnabledTravelAgent_Session2",
            instructions="""
            You are a personalized travel assistant.
            Always address the user by name when you know it.
            Tailor your suggestions based on the user's preferences.
            """,
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()
            query = "Can you recommend my next travel destination?"
            print(f"User: {query}")
            response = await agent.run(query, session=session)
            print(f"Agent: {response.text}\n")

            print("=" * 60)
            print("🎉 Success!")
            print("=" * 60)
            print("Notice that the agent:")
            print("1. Used Sarah's name")
            print("2. Recommended beach destinations")
            print("3. Considered moderate budget")
            print("4. Mentioned vegetarian food options")
            print("\nAll from a different session!")


await agent_with_restored_memory(saved_memory)

## Step 7: Memory Updates Over Time

Memory can evolve as the user shares more information.

In [ ]:
async def memory_evolution():
    """
    Demonstrates how memory grows and updates over time.
    """
    print("=== Memory Evolution Over Time ===")

    async with DefaultAzureCredential() as credential:
        conversation_client = create_azure_openai_chat_client(credential)
        memory_client = create_memory_extraction_client(credential)
        memory = TravelMemory(memory_client)

        async with Agent(
            client=conversation_client,
            name="EvolvingMemoryAgent",
            instructions="You are a helpful travel assistant.",
            context_providers=[memory],
        ) as agent:
            session = agent.create_session()

            print("\n--- Week 1: First Trip ---")
            await agent.run("My name is Tom. I love hiking in the mountains.", session=session)
            print(f"Memory: {memory.profile.model_dump()}\n")

            print("--- Week 2: Discovering More ---")
            await agent.run("I also love historical and cultural landmarks.", session=session)
            print(f"Memory: {memory.profile.model_dump()}\n")

            print("--- Week 3: Budget Planning ---")
            await agent.run("This time I'd like a slightly luxurious experience.", session=session)
            print(f"Memory: {memory.profile.model_dump()}\n")

            print("--- Week 4: Broadening Horizons ---")
            await agent.run("This time I'd also like to try a city break.", session=session)
            print(f"Memory: {memory.profile.model_dump()}\n")

            print("🎯 Memory evolved from basic to a rich user profile!")


await memory_evolution()

## Step 8: Practical Example - Multi-Session Planning

Real-world scenario: A user plans a trip across multiple sessions.

In [ ]:
# Add tools from previous tutorials
def get_weather(
    location: Annotated[str, Field(description="City or country name")],
) -> str:
    """Returns the current weather for the specified location."""
    conditions = ["Sunny", "Partly cloudy", "Cloudy", "Rainy"]
    temp = randint(15, 32)
    return f"{location} weather: {choice(conditions)}, {temp}°C"


def get_attractions(
    location: Annotated[str, Field(description="City or country name")],
) -> str:
    """Returns the main attractions for a destination."""
    attractions = {
        "Bali": "Beaches, rice terraces, temples, surfing, yoga retreats",
        "Thailand": "Bangkok temples, beaches, markets, street food, islands",
        "Maldives": "Beaches, diving, luxury resorts, water sports",
    }
    for city, attrs in attractions.items():
        if city.lower() in location.lower():
            return f"{location}'s main attractions: {attrs}"
    return f"{location}'s popular attractions: Various tourist sites"


print("✅ Tools defined")

In [ ]:
async def multi_session_planning():
    """
    Realistic multi-session travel planning with persistent memory.
    """
    print("=== Multi-Session Travel Planning ===")

    print("\n--- Session 1: Initial Exploration ---")
    async with DefaultAzureCredential() as credential:
        conversation_client = create_azure_openai_chat_client(credential)
        memory_client = create_memory_extraction_client(credential)
        memory = TravelMemory(memory_client)

        async with Agent(
            client=conversation_client,
            name="TravelPlanner",
            instructions="""
            You are a personalized travel planner.
            Use what you know about the user to make customized suggestions.
            """,
            tools=[get_weather, get_attractions],
            context_providers=[memory],
        ) as agent:
            session1 = agent.create_session()

            query1 = "Hi, I'm Emma. I'd like to plan a relaxing beach vacation. I'm vegetarian."
            print(f"User: {query1}")
            response = await agent.run(query1, session=session1)
            print(f"Agent: {response.text}\n")

        session1_memory = memory.serialize()
        print("💾 Saved memory after Session 1")
        print(f"Memory state: {json.loads(session1_memory)}\n")

        print("--- Session 2: Destination Research (3 days later) ---")
        conversation_client = create_azure_openai_chat_client(credential)
        memory_client = create_memory_extraction_client(credential)
        profile = TravelProfile.model_validate(json.loads(session1_memory))
        memory = TravelMemory(memory_client, profile=profile)
        print(f"✅ Memory restored: name={profile.name}, preferences={profile.preferred_destinations}")

        async with Agent(
            client=conversation_client,
            name="TravelPlanner",
            instructions="""
            You are a personalized travel planner.
            Use what you know about the user to make customized suggestions.
            """,
            tools=[get_weather, get_attractions],
            context_providers=[memory],
        ) as agent:
            session2 = agent.create_session()

            query2 = "I'm torn between Bali and the Maldives. What's the weather like?"
            print(f"\nUser: {query2}")
            response = await agent.run(query2, session=session2)
            print(f"Agent: {response.text}\n")

            query3 = "My budget is moderate. Which one do you recommend?"
            print(f"User: {query3}")
            response = await agent.run(query3, session=session2)
            print(f"Agent: {response.text}\n")

        session2_memory = memory.serialize()
        print("💾 Saved memory after Session 2")
        print(f"Memory state: {json.loads(session2_memory)}\n")

        print("--- Session 3: Final Decision (1 week later) ---")
        conversation_client = create_azure_openai_chat_client(credential)
        memory_client = create_memory_extraction_client(credential)
        profile = TravelProfile.model_validate(json.loads(session2_memory))
        memory = TravelMemory(memory_client, profile=profile)
        print(f"✅ Memory restored: name={profile.name}, budget={profile.budget_level}")

        async with Agent(
            client=conversation_client,
            name="TravelPlanner",
            instructions="""
            You are a personalized travel planner.
            Use what you know about the user to make customized suggestions.
            """,
            tools=[get_weather, get_attractions],
            context_providers=[memory],
        ) as agent:
            session3 = agent.create_session()

            query4 = "I've decided on Bali! What are the must-see attractions?"
            print(f"\nUser: {query4}")
            response = await agent.run(query4, session=session3)
            print(f"Agent: {response.text}\n")

    print("=" * 60)
    print("🎉 Multi-session planning complete!")
    print("=" * 60)
    print("Notice:")
    print("- 3 different sessions (days apart)")
    print("- 3 different sessions (separate conversations)")
    print("- Agent remembered: Emma's name, beach preference, vegetarian, budget")
    print("- Memory persisted across all sessions!")


await multi_session_planning()

## 🔍 Understanding Context Providers

### Context Provider Lifecycle

```python
# 1. User sends a message
await agent.run("I love beaches", session=session)

# 2. Before AI call: before_run() is called
await memory.before_run(agent=agent, session=session, context=context, state=state)
# Inject additional instructions via context.extend_instructions()

# 3. AI receives enhanced instructions
# Original: "You are a travel assistant"
# Enhanced: "You are a travel assistant. User preferred destinations: beaches."

# 4. AI responds with personalized answer

# 5. After AI call: after_run() is called
await memory.after_run(agent=agent, session=session, context=context, state=state)
# Get user messages from context.input_messages
# Extract: beach preference -> save to memory
```

### Why Context Providers Are Powerful

- **Application manages state** - Context providers use data persisted by the application
- **Agents are stateless** - Agent definitions are ephemeral, but data persists
- **Flexible context injection** - Add as much context as needed via `context.extend_instructions()` / `context.extend_messages()`
- **Debuggable** - You can see what context was injected

### Production Use Cases

- User profiles (name, preferences, permissions)
- Conversation history summaries
- Recent operation logs
- Business rules and policies
- Shared app state (cart, selected plan, etc.)


## 📝 Key Takeaways

### What We Learned

1. **Thread ≠ Memory**
   - Thread: Conversation history (ephemeral)
   - Memory: User knowledge (persistent)

2. **Context Provider (`BaseContextProvider`)**
   - Listen to conversations (`after_run()`)
   - Provide context (`before_run()`)
   - Persist state (`serialize()`)

3. **Personalization**
   - Agent remembers users across sessions
   - Customized recommendations based on preferences
   - Understanding evolves over time

4. **Practical Patterns**
   - Save memory to database
   - Restore memory at session start
   - Update memory as user shares more

### Best Practices

1. Use **Pydantic models** for structured memory
2. **Serialize frequently** to avoid data loss
3. **Validate extracted data** before saving
4. **Combine with threads** for best results
5. **Handle errors gracefully** during extraction

### Production Patterns

```python
# Database integration
class UserMemoryService:
    async def load_memory(self, user_id: str) -> TravelProfile:
        data = await db.get_user_profile(user_id)
        return TravelProfile.model_validate(data)
    
    async def save_memory(self, user_id: str, memory: TravelMemory):
        await db.update_user_profile(user_id, memory.serialize())

# Usage
profile = await memory_service.load_memory(user_id)
memory = TravelMemory(chat_client, profile=profile)

# ... use agent ...

await memory_service.save_memory(user_id, memory)
```


## 💪 Exercises

1. **Extend TravelProfile** - Add fields like preferred_activities, travel_companions, accessibility_needs
2. **Multi-User Memory** - Create a memory system that handles multiple users
3. **Memory Decay** - Implement time-based memory relevance (older preferences become less important)
4. **Conflict Resolution** - Handle conflicting preferences ("You said you hate cold, but now you want to go to Alaska?")


In [ ]:
# Exercise: Create an enhanced profile with activity preferences

class EnhancedTravelProfile(BaseModel):
    """Enhanced travel profile with more details."""
    # Write your code here - add more fields!
    pass

# Create a context provider that uses this enhanced profile
# Test with various user inputs

## 🚀 Next Steps

Great! You've mastered persistent memory and personalization.

---

### Quick Reference

**Creating a Context Provider:**
```python
class MyMemory(BaseContextProvider):
    async def after_run(self, *, agent, session, context, state):
        # Extract and save information from context.input_messages
        pass
    
    async def before_run(self, *, agent, session, context, state):
        # Inject context
        context.extend_instructions(self.source_id, "User preferences: ...")
```

**Using with an Agent:**
```python
memory = MyMemory(chat_client, source_id="my_memory")
agent = Agent(..., context_providers=[memory])
```

**Persisting Memory:**
```python
saved = memory.serialize()
# ... save to database ...
# ... later ...
restored = MyMemory(chat_client, **json.loads(saved))
```


---

# 🧠 Bonus: Memory Persistence with Azure AI Foundry Memory Store

## Overview

In the previous section, we created custom memory by inheriting from `BaseContextProvider`.
Here, we'll test using the **managed memory store provided by Azure AI Foundry** (`FoundryMemoryProvider`)
to **automatically** save, search, and update memories **server-side**.

### Features of FoundryMemoryProvider

| Feature | Description |
|---|---|
| **Automatic User Profile Extraction** | Automatically extracts and accumulates user information from conversations |
| **Semantic Search** | Automatically searches for and injects memories related to input messages |
| **Scope Isolation** | Isolates memory per user with the `scope` parameter |
| **Deferred Batch Updates** | Cost optimization with `update_delay` (default 5 minutes) |

### Prerequisites

- Azure AI Foundry project created
- Chat model (gpt-4.1-mini, etc.) deployed
- **Embedding model** (text-embedding-3-small, etc.) deployed
- `az login` completed

### Official Sample

`agent-framework-python-1.0.0rc2/python/samples/02-agents/context_providers/azure_ai_foundry_memory.py`


## Step B1: Additional Imports and Setup for Foundry Memory


In [ ]:
from datetime import datetime, timezone

from agent_framework import InMemoryHistoryProvider
from agent_framework.azure import AzureOpenAIResponsesClient, FoundryMemoryProvider
from azure.ai.projects.aio import AIProjectClient
from azure.ai.projects.models import (
    MemoryStoreDefaultDefinition,
    MemoryStoreDefaultOptions,
)

# Deployment name settings (loaded from .env)
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME", "text-embedding-3-small")
responses_deployment = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME") or model_deployment

print("✅ Foundry Memory imports complete")
print(f"   Project Endpoint: {project_endpoint}")
print(f"   Chat/Responses Model: {responses_deployment}")
print(f"   Embedding Model: {embedding_deployment}")

if not os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"):
    print("ℹ️ AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME is not set. Using AZURE_AI_MODEL_DEPLOYMENT_NAME instead.")

## Step B2: Create a Memory Store

Create a memory store in Azure AI Foundry.  
- `user_profile_enabled=True`: Enables automatic user profile extraction  
- `chat_summary_enabled=False`: Disables chat summary (for demo purposes)

Memory store names must be unique, so we generate a name with a timestamp.


In [ ]:
embedding_deployment

In [ ]:
project_endpoint = "https://foundry-sweden-central-hana1.services.ai.azure.com/api/projects/proj-default"

In [ ]:
# Create memory store
memory_store_name = f"workshop_memory_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME", "text-embedding-3-small")
responses_deployment = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME") or model_deployment

# Memory store options
options = MemoryStoreDefaultOptions(
    chat_summary_enabled=False,       # Disable chat summary
    user_profile_enabled=True,        # Enable user profile extraction
    user_profile_details="Extract travel-related information such as travel preferences, dietary restrictions, budget level, and interests."
    "Avoid sensitive data such as age, financial information, exact addresses, and credentials.",
)

memory_store_definition = MemoryStoreDefaultDefinition(
    chat_model=responses_deployment,
    embedding_model=embedding_deployment,
    options=options,
)

# Create memory store using AIProjectClient
async with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
):
    print(f"Creating memory store '{memory_store_name}'...")
    memory_store = await project_client.memory_stores.create(
        name=memory_store_name,
        description="Agent Framework Workshop: FoundryMemoryProvider test",
        definition=memory_store_definition,
    )

    print("✅ Memory store created!")
    print(f"   Name: {memory_store.name}")
    print(f"   ID: {memory_store.id}")
    print(f"   Description: {memory_store.description}")

## Step B3: Running an Agent with FoundryMemoryProvider

We'll integrate `FoundryMemoryProvider` into the agent following the same pattern as the official sample.

**Key Points:**
- `scope="user_123"` — Isolates memory by user ID
- `update_delay=0` — Updates memory immediately for demo (300 seconds recommended for production)
- `InMemoryHistoryProvider(load_messages=False)` — Intentionally don't load conversation history
  - This allows us to verify that the agent relies solely on Foundry memory for responses
- `default_options={"store": False}` — Don't store conversation history on the service side either


In [ ]:
async def test_foundry_memory():
    """
    Test semantic memory with FoundryMemoryProvider.
    Following the official sample azure_ai_foundry_memory.py.
    """
    async with (
        DefaultAzureCredential() as credential,
        AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
    ):
        # Create chat client (explicitly specifying deployment_name)
        client = AzureOpenAIResponsesClient(
            project_client=project_client,
            deployment_name=responses_deployment,
        )

        # Create FoundryMemoryProvider
        memory_provider = FoundryMemoryProvider(
            project_client=project_client,
            memory_store_name=memory_store_name,
            scope="user_123",    # Isolate memory by user ID
            update_delay=0,      # Demo: update memory immediately (300 seconds recommended for production)
        )

        # Create agent with memory
        async with Agent(
            name="FoundryMemoryAgent",
            client=client,
            instructions="""You are a friendly assistant knowledgeable about travel.
                Memories from past conversations are automatically provided.
                Always address the user by name when you know it.
                Make personalized suggestions based on the user's preferences.""",
            context_providers=[
                memory_provider,
                InMemoryHistoryProvider(load_messages=False),  # Disable history loading (test memory only)
            ],
            default_options={"store": False},  # Don't store conversation history on service side either
        ) as agent:
            session = agent.create_session()

            # === Conversation 1: Sharing user preferences ===
            print("=" * 60)
            print("=== Conversation 1: Sharing User Preferences ===")
            print("=" * 60)
            query1 = "My name is Haruka. I love beach resorts and I'm vegetarian."
            print(f"User: {query1}")
            result1 = await agent.run(query1, session=session)
            print(f"Agent: {result1.text}\n")

            # Wait for memory to be processed
            print("⏳ Waiting for memory to be saved to store (8 seconds)...")
            await asyncio.sleep(8)

            # === Conversation 2: Test recall from memory ===
            print("=" * 60)
            print("=== Conversation 2: Testing Recall from Memory ===")
            print("=" * 60)
            query2 = "Can you recommend restaurants for my next travel destination?"
            print(f"User: {query2}")
            result2 = await agent.run(query2, session=session)
            print(f"Agent: {result2.text}\n")

            # === Conversation 3: Verify memory contents ===
            print("=" * 60)
            print("=== Conversation 3: Verifying Memory Contents ===")
            print("=" * 60)
            query3 = "Tell me what you remember about me."
            print(f"User: {query3}")
            result3 = await agent.run(query3, session=session)
            print(f"Agent: {result3.text}\n")

        # === Directly check memory store contents ===
        print("=" * 60)
        print("📦 Memories saved in Foundry Memory Store:")
        print("=" * 60)
        res = await project_client.memory_stores.search_memories(
            name=memory_store_name, scope="user_123"
        )
        for i, memory in enumerate(res.memories, 1):
            print(f"  {i}. {memory.memory_item.content}")

        if not res.memories:
            print("  (No memories saved yet)")


await test_foundry_memory()

## Step B4: Cleanup — Delete the Memory Store

After testing, delete the memory store to clean up resources.  
**Note**: Deleting will permanently remove all saved memories.


In [ ]:
# Delete memory store (cleanup)
async with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=project_endpoint, credential=credential) as project_client,
):
    await project_client.memory_stores.delete(memory_store_name)
    print(f"🗑️ Deleted memory store '{memory_store_name}'.")

## 🔍 FoundryMemoryProvider vs Custom TravelMemory Comparison

| Aspect | Custom TravelMemory (First Half) | FoundryMemoryProvider (Bonus) |
|---|---|---|
| **Memory Storage** | Application-side (variables / JSON) | Azure AI Foundry server-side |
| **Information Extraction** | Custom LLM calls for structured extraction | Foundry automatically extracts profiles |
| **Search Method** | None (always injects all memory) | Semantic search (retrieves only relevant memories) |
| **Scalability** | Requires DB integration | Auto-scales with managed service |
| **Cost** | LLM calls × 2 (extraction + response) | Foundry Memory API usage fee |
| **Customizability** | Fully flexible | Conforms to Foundry specifications |
| **Implementation Effort** | High (write everything yourself) | Low (completed in a few lines) |

### When to Use Which?

- **FoundryMemoryProvider**: When you want to quickly add memory capabilities, or when you're already using Azure AI Foundry
- **Custom BaseContextProvider**: When you want full control over memory structure, or when using storage other than Foundry
